<a href="https://colab.research.google.com/github/Thilac01/Point_Cloud/blob/main/Point_Cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import os

In [ ]:
file_path = "room_scan2.pcd"

if not os.path.exists(file_path):
    print(f"ERROR: '{file_path}' not found. Please upload it to Colab!")
else:
    # Load the point cloud
    pcd = o3d.io.read_point_cloud(file_path)

    if pcd.is_empty():
        print("ERROR: The point cloud is empty or corrupted.")
    else:
        print(f"Success! Loaded {len(pcd.points)} points from {file_path}.")

In [ ]:
# Create a coordinate frame (X=Red, Y=Green, Z=Blue) to help orient the view
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])

print("Rendering original 3D point cloud...")

# Visualize inline
o3d.visualization.draw_plotly(
    [pcd, coordinate_frame],
    window_name="Original Point Cloud",
    width=1000,
    height=600
)

#View All the X,Y,Z corrdinate only

In [ ]:


pcd = o3d.io.read_point_cloud("room_scan1.pcd")

points = np.asarray(pcd.points)
print(points)


print(f"Orignal data size: {len(points)}")

In [ ]:
# Set voxel size (e.g., 0.05 = 5 centimeters if your data is in meters)
voxel_size = 0.8

print(f"Original point count: {len(pcd.points)}")

# Perform downsampling
downpcd = pcd.voxel_down_sample(voxel_size=voxel_size)

print(f"Downsampled point count: {len(downpcd.points)}")

# Calculate the reduction
reduction = 100 - ((len(downpcd.points) / len(pcd.points)) * 100)
print(f"Data reduced by: {reduction:.2f}%")

In [ ]:
print("Rendering downsampled 3D point cloud...")

o3d.visualization.draw_plotly(
    [downpcd, coordinate_frame],
    window_name="Downsampled Point Cloud",
    width=1000,
    height=600
)

In [ ]:
# 1. Convert the downsampled Open3D object into a raw NumPy array
points_3d = np.asarray(downpcd.points)
print(f"3D Array Shape: {points_3d.shape}")

# 2. Initialize PCA to force the data into exactly 2 dimensions
pca = PCA(n_components=2)

# 3. Transform the data
points_2d = pca.fit_transform(points_3d)

print(f"2D Array Shape: {points_2d.shape}")
print(f"Geometry preserved (Variance retained): {sum(pca.explained_variance_ratio_) * 100:.2f}%")

In [ ]:
plt.figure(figsize=(10, 10))

# Scatter plot: X is PCA Component 1, Y is PCA Component 2
# s=1 sets the dot size to be very small for a cleaner map
plt.scatter(points_2d[:, 0], points_2d[:, 1], s=1, c='black', alpha=0.6)

plt.title("2D Floor Plan Projection (Flattened via PCA)")
plt.xlabel("Component 1 (Length/Width)")
plt.ylabel("Component 2 (Length/Width)")

# KEEP THIS: It ensures your room doesn't look stretched out like a funhouse mirror
plt.axis('equal')
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

#TEST 1

In [ ]:
import alphashape
from shapely.geometry import Polygon, MultiPolygon
from rdp import rdp
import numpy as np
import matplotlib.pyplot as plt

# 1. Create Alpha Shape
alpha = 0.5
alpha_shape = alphashape.alphashape(points_2d, alpha)

# 2. Fix: Check if it's a MultiPolygon or Polygon
if isinstance(alpha_shape, MultiPolygon):
    # If there are multiple parts, grab the one with the largest area
    # This ignores small noisy 'islands' that might have been detected
    alpha_shape = max(alpha_shape.geoms, key=lambda a: a.area)

# 3. Get exterior coordinates
hull_pts = np.array(alpha_shape.exterior.coords)

# 4. Simplify: Douglas-Peucker algorithm
corners = rdp(hull_pts, epsilon=0.8)

# 5. Plot
plt.figure(figsize=(10, 10))
plt.scatter(points_2d[:, 0], points_2d[:, 1], s=1, c='gray', alpha=0.3)
plt.plot(np.array(corners)[:, 0], np.array(corners)[:, 1], 'r-', linewidth=2, label="Wall Perimeter")
plt.scatter(np.array(corners)[:, 0], np.array(corners)[:, 1], s=100, c='red', label="Corner Points")

plt.title("Robust Irregular Room Corner Detection")
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

#TEST 2

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Rasterize to a binary image
# We create a 500x500 image representing your floor area
res = 500
img = np.zeros((res, res), dtype=np.uint8)
pts_norm = ((points_2d - points_2d.min(axis=0)) / (points_2d.max(axis=0) - points_2d.min(axis=0)) * (res-1)).astype(int)
for p in pts_norm:
    img[p[1], p[0]] = 255

# 2. Morphological Closing: This "connects" the wall points into solid lines
# kernel size (15,15) determines how wide to bridge the gaps
kernel = np.ones((15, 15), np.uint8)
closed = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

# 3. Find the main outer contour
contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
# Get the largest contour
main_contour = max(contours, key=cv2.contourArea)

# 4. Simplify the contour to get corners (approxPolyDP)
# epsilon is a % of the perimeter; decrease for more corners, increase for fewer
epsilon = 0.005 * cv2.arcLength(main_contour, True)
corners = cv2.approxPolyDP(main_contour, epsilon, True)

# 5. Plot
plt.figure(figsize=(10, 10))
plt.scatter(points_2d[:, 0], points_2d[:, 1], s=1, c='lightgray')
# Map back to real coordinates
c_real = (corners.reshape(-1, 2) / (res-1)) * (points_2d.max(axis=0) - points_2d.min(axis=0)) + points_2d.min(axis=0)
plt.plot(np.vstack((c_real, c_real[0]))[:, 0], np.vstack((c_real, c_real[0]))[:, 1], 'r-', linewidth=3)
plt.scatter(c_real[:, 0], c_real[:, 1], s=150, c='red')
plt.axis('equal'); plt.grid(True); plt.show()

#Test 03

In [ ]:
import alphashape
from shapely.geometry import Polygon, MultiPolygon
from rdp import rdp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

# 1. CLEANING: Statistical Outlier Removal
# We keep points that have at least 15 neighbors within a 0.5 radius
# This naturally removes the 'floating' noise without needing density maps
nbrs = NearestNeighbors(n_neighbors=15, radius=0.5).fit(points_2d)
distances, _ = nbrs.kneighbors(points_2d)
mean_distances = np.mean(distances, axis=1)
# Keep only points where mean distance is below a threshold
clean_points = points_2d[mean_distances < np.percentile(mean_distances, 90)]

# 2. ALPHA SHAPE: Generate the footprint
alpha = 0.5
alpha_shape = alphashape.alphashape(clean_points, alpha)

# Fix for MultiPolygon
if isinstance(alpha_shape, MultiPolygon):
    alpha_shape = max(alpha_shape.geoms, key=lambda a: a.area)
elif isinstance(alpha_shape, Polygon) == False:
    # Fallback if alpha shape returns a geometry collection
    alpha_shape = alpha_shape.convex_hull

# 3. SIMPLIFY: Douglas-Peucker
# Increase epsilon slightly if the perimeter is still too jagged
hull_pts = np.array(alpha_shape.exterior.coords)
corners = rdp(hull_pts, epsilon=1.2)

# 4. PLOT
plt.figure(figsize=(10, 10))
plt.scatter(clean_points[:, 0], clean_points[:, 1], s=1, c='gray', alpha=0.3, label="Cleaned Points")
corners_arr = np.array(corners)
plt.plot(corners_arr[:, 0], corners_arr[:, 1], 'r-', linewidth=3, label="Wall Perimeter")
plt.scatter(corners_arr[:, 0], corners_arr[:, 1], s=150, c='red', label="Corner Points")

plt.title("Cleaned Room Corner Detection")
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

#Test 04

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize

# 1. Rasterize to high-res image
res = 800
img = np.zeros((res, res), dtype=np.uint8)
pts_norm = ((points_2d - points_2d.min(axis=0)) / (points_2d.max(axis=0) - points_2d.min(axis=0)) * (res-1)).astype(int)
for p in pts_norm:
    img[p[1], p[0]] = 255

# 2. Closing and Dilation to make walls "solid"
kernel = np.ones((10,10), np.uint8)
solid = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

# 3. Get the Contour (This finds the clean outer wall)
contours, _ = cv2.findContours(solid, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
main_contour = max(contours, key=cv2.contourArea)

# 4. Douglas-Peucker with a tighter tolerance (epsilon)
# This handles irregular shapes by keeping more vertices where needed
epsilon = 0.0015 * cv2.arcLength(main_contour, True)
corners = cv2.approxPolyDP(main_contour, epsilon, True)

# 5. Plotting
plt.figure(figsize=(10, 10))
c_real = (corners.reshape(-1, 2) / (res-1)) * (points_2d.max(axis=0) - points_2d.min(axis=0)) + points_2d.min(axis=0)

# Draw the polygon boundary
plt.fill(c_real[:, 0], c_real[:, 1], 'lightgray', alpha=0.5)
plt.plot(np.vstack((c_real, c_real[0]))[:, 0], np.vstack((c_real, c_real[0]))[:, 1], 'k-', linewidth=2)
plt.scatter(c_real[:, 0], c_real[:, 1], s=80, c='red', zorder=5)

plt.title("Professional Floor Plan Mapping (Contour-based)")
plt.axis('equal'); plt.grid(True); plt.show()

In [ ]:
import pyvista as pv
import numpy as np
from shapely.geometry import Polygon

# Assuming 'c_real' is your array of shape (N, 2) from your previous code
# Ensure it is a closed polygon
poly = Polygon(c_real)

# 1. Create Wall Thickness
# Offset the polygon inward to create the "inner" boundary of the wall
wall_thickness = 0.05 # Adjust as needed (e.g., 0.05 or 0.1 for more realistic thickness)
inner_poly = poly.buffer(-wall_thickness)

# Convert back to vertices
outer_pts = np.array(poly.exterior.coords)[:-1]

# 2. Create the 3D Wall Extrusion
def create_wall_mesh(outer, inner, height=3.0):
    n_outer = len(outer)

    # Outer wall extrusion
    outer_line = pv.lines_from_points(np.column_stack((outer, np.zeros(n_outer))))
    outer_wall = outer_line.extrude((0, 0, height))

    # Inner wall extrusion (only if inner_poly is not empty)
    inner_wall = None
    if not inner_poly.is_empty and inner_poly.exterior is not None:
        inner_pts = np.array(inner_poly.exterior.coords)[:-1]
        n_inner = len(inner_pts)
        if n_inner > 1: # Ensure there are enough points to form a line
            inner_line = pv.lines_from_points(np.column_stack((inner_pts, np.zeros(n_inner))))
            inner_wall = inner_line.extrude((0, 0, height))

    return outer_wall, inner_wall

outer_wall, inner_wall = create_wall_mesh(outer_pts, inner_poly)

# 3. Visualization
pl = pv.Plotter()
pl.add_mesh(outer_wall, color='gray', show_edges=True)
if inner_wall is not None:
    pl.add_mesh(inner_wall, color='gray', show_edges=True)
pl.add_text("3D Floor Plan Walls", font_size=12)
pl.show()